# Introduction

In [13]:
from rdfine import PrefixStore, GraphReader, GraphTable, GraphDict
from rdflib import Graph, URIRef

This is a demo for the package "rdfine". It contains the following classes:

- PrefixStore: Stores prefixes and their urls as key-value pairs. Supports various operations based on prefixes, such as expanding, compacting, dropping or adding prefixes.
- GraphReader: Loads a RDF graph into memory to interact with it. Can execute queries, rename nodes, extract subgraphs. Has RDFlib graph at its core. 
- GraphTable: Expresses a Graph as a table, with a pandas DataFrame at its core. Allows treating graphs as a regular dataframe, but also provides advanced filtering.
- GraphDict: Expresses a Graph as a dictionary, with dict at its core. Allows searching for matching entries by patterns in index-paths and / or values. 

With these classes, rdfine allows to integrate graphs into Pythonic coding patterns: Instead of relying solely on SPARQL to query a graph, you can now turn graph data into dataframes and dictionaries, or extract data from graphs as lists and Python's native data types. This allows you to loop over, filter, transform, separate and merge data from graphs without having to rely on SPARQL for this. Extracted graph data can be readily serialized in different formats such as turtle, JSON or yaml.

##### Load Data

In [3]:
input_folder = "..\\data\\"
graph = Graph()
graph.parse(input_folder + "graph.ttl", publicID="file:///workspace/pipeline/")


<Graph identifier=N10f3a3cae88445ba9125f555b474af0e (<class 'rdflib.graph.Graph'>)>

## GraphTable

In [11]:
# Initialize a new GraphTable based on a Graph
graph_table = GraphTable(graph)

##### Select and Display Data

In [ ]:
# Filter the graph table by matching triples:
graph_table.subset({"sub" : ":InteroperablePipeline", "pred" : "rdf:type"})

# You can also filter by allowing several search terms or by matching by type:
graph_table.subset({"pred" : "rdf:type", "obj" : ["tc:PipelineComponent", "tc:PipelineStep"], "obj_type" : URIRef})

# You can reverse the filter by dropping any triple that matches
graph_table.subset({"pred" : "rdf:type", "obj" : ["tc:PipelineComponent", "tc:PipelineStep"], "obj_type" : URIRef}, action="drop")

# You can also add your own columns and include these in the filter...

,sub,pred,obj,sub_type,obj_type
0,_:n30a67d50d3794dff9b4613bbb62f221cb49,sh:maxCount,1,<class 'rdflib.term.BNode'>,<class 'rdflib.term.Literal'>
1,:RdfcPipeline,rdfs:label,Simple RDF Connect pipeline.,<class 'rdflib.term.URIRef'>,<class 'rdflib.term.Literal'>
2,_:n30a67d50d3794dff9b4613bbb62f221cb58,sh:class,rdfc:Reader,<class 'rdflib.term.BNode'>,<class 'rdflib.term.URIRef'>
3,_:n30a67d50d3794dff9b4613bbb62f221cb47,sh:minCount,1,<class 'rdflib.term.BNode'>,<class 'rdflib.term.Literal'>
4,_:n30a67d50d3794dff9b4613bbb62f221cb28,dcat:hadRole,:configShape,<class 'rdflib.term.BNode'>,<class 'rdflib.term.URIRef'>
...,...,...,...,...,...
289,_:n30a67d50d3794dff9b4613bbb62f221cb51,sh:minCount,0,<class 'rdflib.term.BNode'>,<class 'rdflib.term.Literal'>
290,:LdioHttpOutConfigShape,rdf:type,sh:NodeShape,<class 'rdflib.term.URIRef'>,<class 'rdflib.term.URIRef'>
291,:DishacledCatalog,dcat:resource,sw:loket-error-alert-service,<class 'rdflib.term.URIRef'>,<class 'rdflib.term.URIRef'>
292,rdfc:HttpOut,dcat:qualifiedRelation,_:n30a67d50d3794dff9b4613bbb62f221cb44,<class 'rdflib.term.URIRef'>,<class 'rdflib.term.BNode'>


##### Filter Down Data

In [ ]:
# You can filter down data by using the subset function and overwriting the GraphTable's df with the filtered one
filtered_df = graph_table.subset({"pred" : "rdf:type", "obj" : ["tc:Assignment", "sh:NodeShape"]}, action="drop")
graph_table.df = filtered_df

##### Transform data

In [ ]:
# Transform data by using the native pandas .loc -functionality
graph_table.df.loc[graph_table.df["sub"] == ":InteroperablePipeline", "sub"] = ":NewName"
graph_table.df.loc[graph_table.df["obj"] == ":InteroperablePipeline", "obj"] = ":NewName"

##### Serialize data

In [ ]:
# You can return the GraphTable as graph and let RDFlib handle the serialization

output_graph = graph_table.to_graph()
print(output_graph.serialize(format = "turtle"))

##### Other

In [ ]:
# add_triples

# to be continued...

## GraphReader

In [4]:
# Initialize a new GraphReader based on a Graph
graph_reader = GraphReader(graph)

##### Select and Display Data

In [ ]:
# Show all triples as a table:
graph_reader.get_triples()

# Show triples matching a pattern:
graph_reader.get_triples(pred = "rdf:type")
graph_reader.get_triples(sub = ":InteroperablePipeline")

# Execute a query
# A select query returns results as a dataframe
select_query = f"""
            SELECT ?step ?prev_step?component
            WHERE {{
            ?step p-plan:isStepOfPlan :InteroperablePipeline .
            OPTIONAL {{?step p-plan:isPrecededBy ?prev_step .}}
            ?step p-plan:hasInputVar ?assignment .
            ?assignment tc:component ?component .
            }}
        """
graph_reader.execute_query(select_query)

,step,prev_step,component
0,:LdioHttpInStep,:RdfcHttpOutStep,ldio:HttpIn
1,:RdfcHttpOutStep,:RdfcPollApiStep,rdfc:HttpOut
2,:LdioConsoleOutStepInter,:LdioHttpInStep,ldio:ConsoleOut
3,:RdfcPollApiStep,None,rdfc:HttpFetch


##### Filter down data

In [ ]:
# You can extract subgraphs from the GraphReader via graph traversal:
subgraph = graph_reader.extract_subgraph(
            ":InteroperablePipeline", direction="along", against="p-plan:isStepOfPlan", prune=["dcat:qualifiedRelation", "dct:requires"]
        )

# In this example, a subgraph is extracted from the node with uri ":InteroperablePipeline". 
# From there, it follows all edges originating from this node (direction="along") recursively. 
# "dcat:qualifiedRelation" and "dct:requires" will also be followed in the default direction, 
# however graph traversal is stopped at the destination of these edges and hence not recursively repeated (prune=["dcat:qualifiedRelation", "dct:requires"])
# The only exeception here are triples with the predicate p-plan:isStepOfPlan, where :InteroperablePipeline is the object of the triple.
# These will also be followed (against="p-plan:isStepOfPlan").
# You can see that "extract_subgraph" allows to extract subsets of data in bigger graph by defining which edges should be followed in 
# which direction, given a starting node. 

##### Transform data

In [ ]:
# A construct query returns results as a rdflib Graph. This graph can be turned into a new graph_reader
construct_query = f"""
            CONSTRUCT 
            {{ 
                ?step :follows ?prev_step . 
                ?step :uses ?component .
            }}
            WHERE {{
            ?step p-plan:isStepOfPlan :InteroperablePipeline .
            OPTIONAL {{?step p-plan:isPrecededBy ?prev_step .}}
            ?step p-plan:hasInputVar ?assignment .
            ?assignment tc:component ?component .
            }}
        """

constructed_graph = graph_reader.execute_query(construct_query)
constructed_graph_reader = GraphReader(constructed_graph)
constructed_graph_reader.get_triples()

# Rename a URI to different one in the graph:
graph_reader.rename(":InteroperablePipeline", ":NewName")

# You can also rename nodes via match patterns. This is ideal to target blind nodes and turn them to proper Uris
graph_reader.rename(":InteroperablePipeline tc:config ?target.", ":LdioPipelineConfig")

##### Serialize data

In [ ]:
# Data is serialized with the GraphReader by simply turning it back to a graph and letting RDFlib handle the serializatin
output_graph = graph_reader.to_graph()
print(output_graph.serialize(format="turtle"))

##### Other

In [ ]:
# Check whether a specific URI exists in the graph:
graph_reader.check_node_exists(":InteroperablePipeline")

<Graph identifier=N326889cf9f7840bc9406d48147f87d86 (<class 'rdflib.graph.Graph'>)>

## PrefixStore

In [ ]:
# To be continued...